# BIL545 - Proje 1
## Banka Pazarlamasında Abonelik Tahmini

### Bölüm 1 - Veri, Ön İşleme ve Veri Sızıntısı

Bu çalışmada UCI Bank Marketing veri kümesi
kullanılarak müşterilerin vadeli mevduata abonelik
durumları incelenecektir.

Amaç, veri kümesini tanımak, veri ön işleme
yöntemlerini değerlendirmek ve veri sızıntısının
model sonuçlarına etkisini araştırmaktır.

In [1]:
import sys
import random
import numpy as np
import pandas as pd
import sklearn
import scipy

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("SciPy:", scipy.__version__)
print("Random seed:", SEED)

Python: 3.13.18
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1
SciPy: 1.16.3
Random seed: 42


## 1. Veri Kümesinin Yüklenmesi

Bu projede UCI Machine Learning Repository üzerinden
sağlanan Bank Marketing veri kümesi kullanılmaktadır.

Veri kümesi, Portekiz'deki bir bankanın telefon
pazarlama kampanyalarına ait kayıtları içermektedir.

Hedef değişken `y`, müşterinin vadeli mevduata
abone olup olmadığını göstermektedir.

Veri dosyası GitHub deposuna eklenmeyecek,
Google Colab ortamına ayrıca yüklenecektir.

In [2]:
from pathlib import Path
from google.colab import files

# Veri dosyasının konumu
dosya_yolu = Path("/content/bank-full.csv")

# Dosya bulunmuyorsa kullanıcıdan yüklemesini iste
if not dosya_yolu.exists():
    print("Lütfen bank-full.csv dosyasını yükleyin.")
    files.upload()

# Dosyanın varlığını kontrol et
if not dosya_yolu.exists():
    raise FileNotFoundError("bank-full.csv bulunamadı.")

# Veri kümesini oku
df = pd.read_csv(dosya_yolu, sep=";")

# Temel bilgileri göster
print("Satır sayısı:", df.shape[0])
print("Sütun sayısı:", df.shape[1])
print("Sütunlar:", df.columns.tolist())

display(df.head())

Lütfen bank-full.csv dosyasını yükleyin.


Saving bank-full.csv to bank-full.csv
Satır sayısı: 45211
Sütun sayısı: 17
Sütunlar: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Keşifsel Veri Analizi (EDA)

Bu bölümde veri kümesinin yapısı, değişken türleri,
hedef değişkenin dağılımı ve yinelenen kayıtlar
incelenecektir.

Amaç, model geliştirmeden önce verinin temel
özelliklerini ve olası problemlerini belirlemektir.

In [3]:
# Veri kümesinin boyutları
print("Veri kümesinin boyutu:", df.shape)

# Değişkenlerin veri tipleri
print("\nDeğişken tipleri:")
display(df.dtypes.to_frame(name="Veri tipi"))

# Hedef değişkenin dağılımı
print("\nHedef değişken dağılımı:")

hedef_dagilimi = pd.DataFrame({
    "Adet": df["y"].value_counts(),
    "Oran (%)": df["y"].value_counts(normalize=True) * 100
})

display(hedef_dagilimi.round(2))

# Yinelenen satır kontrolü
print("\nYinelenen satır sayısı:", df.duplicated().sum())

Veri kümesinin boyutu: (45211, 17)

Değişken tipleri:


,Veri tipi
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64



Hedef değişken dağılımı:


,Adet,Oran (%)
y,,
no,39922,88.3
yes,5289,11.7



Yinelenen satır sayısı: 0


### Yorum - İlk Veri İncelemesi

Veri kümesinde 45.211 gözlem ve 17 değişken bulunmaktadır.
Bunların 16'sı girdi değişkeni, biri hedef değişken olan
y sütunudur.

Değişkenlerin 7'si int64, 10'u object veri tipindedir.
Ancak bu tipler değişkenlerin ölçüm ölçeklerini
doğrudan göstermemektedir.

Hedef değişken incelendiğinde kayıtların %88,30'unun
abonelikle sonuçlanmadığı, %11,70'inin ise abonelikle
sonuçlandığı görülmektedir.

Bu dağılım, sınıflar arasında belirgin bir dengesizlik
olduğunu göstermektedir. Bu nedenle model
değerlendirmesinde yalnızca doğruluk (accuracy)
ölçütü yeterli olmayacaktır.

Veri kümesinde birebir yinelenen satır bulunmamıştır.
Bununla birlikte müşteri kimlikleri mevcut olmadığından
aynı müşterinin farklı kampanya kayıtlarında
bulunup bulunmadığı belirlenememektedir.

## 3. Öznitelik Bazında Veri Kalitesi Analizi

Bu bölümde her değişkenin saklama tipi, eksik değer oranı,
farklı değer sayısı ve sıfır oranı incelenmektedir.

Amaç, veri ön işleme aşamasından önce olası veri kalitesi
problemlerini belirlemektir.

In [4]:
# Her öznitelik için veri kalitesi bilgileri
kalite_sonuclari = []

for sutun in df.columns:

    # Sütunun saklama tipi
    veri_tipi = str(df[sutun].dtype)

    # Gerçek NaN değerlerinin oranı
    eksik_orani = df[sutun].isna().mean() * 100

    # Farklı değer sayısı
    farkli_deger = df[sutun].nunique(dropna=True)

    # Sıfır oranı (yalnızca sayısal değişkenlerde)
    if pd.api.types.is_numeric_dtype(df[sutun]):
        sifir_orani = (df[sutun] == 0).mean() * 100
    else:
        sifir_orani = float("nan")

    kalite_sonuclari.append({
        "Öznitelik": sutun,
        "Saklama tipi": veri_tipi,
        "Eksik oranı (%)": eksik_orani,
        "Farklı değer sayısı": farkli_deger,
        "Sıfır oranı (%)": sifir_orani
    })

# Sonuçları tabloya dönüştür
kalite_tablosu = pd.DataFrame(kalite_sonuclari)

display(kalite_tablosu.round(2))

,Öznitelik,Saklama tipi,Eksik oranı (%),Farklı değer sayısı,Sıfır oranı (%)
0,age,int64,0.0,77,0.00
1,job,object,0.0,12,NaN
2,marital,object,0.0,3,NaN
3,education,object,0.0,4,NaN
4,default,object,0.0,2,NaN
5,balance,int64,0.0,7168,7.77
6,housing,object,0.0,2,NaN
7,loan,object,0.0,2,NaN
8,contact,object,0.0,3,NaN
9,day,int64,0.0,31,0.00


In [5]:
# Kategorik sütunları belirle
kategorik_sutunlar = df.select_dtypes(
    include=["object"]
).columns

print("UNKNOWN DEĞERLERİ")
print("-" * 45)

for sutun in kategorik_sutunlar:

    adet = (df[sutun] == "unknown").sum()
    oran = adet / len(df) * 100

    if adet > 0:
        print(
            f"{sutun}: {adet} kayıt (%{oran:.2f})"
        )


print("\nPDAYS ÖZEL DEĞER KONTROLÜ")
print("-" * 45)

pdays_eksi_bir = (df["pdays"] == -1).sum()

print("pdays = -1:", pdays_eksi_bir)
print(
    "Oran:",
    round(pdays_eksi_bir / len(df) * 100, 2),
    "%"
)


print("\nPREVIOUS SIFIR KONTROLÜ")
print("-" * 45)

previous_sifir = (df["previous"] == 0).sum()

print("previous = 0:", previous_sifir)

# İki değişkenin ilişkisini incele
display(
    pd.crosstab(
        df["pdays"] == -1,
        df["previous"] == 0,
        rownames=["pdays = -1"],
        colnames=["previous = 0"]
    )
)

UNKNOWN DEĞERLERİ
---------------------------------------------
job: 288 kayıt (%0.64)
education: 1857 kayıt (%4.11)
contact: 13020 kayıt (%28.80)
poutcome: 36959 kayıt (%81.75)

PDAYS ÖZEL DEĞER KONTROLÜ
---------------------------------------------
pdays = -1: 36954
Oran: 81.74 %

PREVIOUS SIFIR KONTROLÜ
---------------------------------------------
previous = 0: 36954


previous = 0,False,True
pdays = -1,,
False,8257,0
True,0,36954


In [6]:
# Önceki iletişim durumu ve kampanya sonucu
kontrol = pd.crosstab(
    df["pdays"] == -1,
    df["poutcome"] == "unknown",
    rownames=["pdays = -1"],
    colnames=["poutcome = unknown"]
)

display(kontrol)

# Önceden iletişim kurulmuş ancak
# kampanya sonucu bilinmeyen kayıtlar
supheli_kayitlar = df.loc[
    (df["previous"] > 0) &
    (df["poutcome"] == "unknown"),
    ["previous", "pdays", "poutcome"]
]

print("İncelenecek kayıt sayısı:", len(supheli_kayitlar))

display(supheli_kayitlar.head(10))

poutcome = unknown,False,True
pdays = -1,,
False,8252,5
True,0,36954


İncelenecek kayıt sayısı: 5


,previous,pdays,poutcome
40658,1,98,unknown
41821,5,168,unknown
42042,2,188,unknown
43978,2,416,unknown
45021,7,528,unknown


### Yorum – Veri Kalitesi ve Özel Değerler

Veri kümesinde doğrudan NaN olarak kaydedilmiş
eksik değer bulunmamaktadır. Ancak bazı kategorik
değişkenlerde "unknown" değerleri görülmektedir.

Özellikle contact (%28,80) ve poutcome (%81,75)
değişkenlerinde unknown oranları yüksektir.

previous değişkeninin %81,74'ü sıfırdır.
pdays değişkeninde ise aynı oranda -1 değeri
bulunmaktadır.

Çapraz tablo analizi sonucunda previous=0 ve
pdays=-1 durumlarının bütün kayıtlarda örtüştüğü
belirlenmiştir. Bu nedenle pdays=-1 değeri doğrudan
hatalı veya rastgele eksik veri olarak yorumlanamaz.

poutcome=unknown olan kayıtların beşinde önceki
iletişim sayısı sıfırdan büyüktür. Dolayısıyla
unknown değeri yalnızca önceki iletişimin
bulunmamasını temsil etmemektedir.

Bu bulgular, eksik değerlerin dönüştürülmesinden
önce değişkenlerin anlamlarının incelenmesi
gerektiğini göstermektedir.

Eksiklik mekanizmasının MCAR, MAR veya MNAR
olduğu yalnızca bu sayımlardan belirlenemez.

## 4. Değişkenlerin Ölçüm Ölçekleri

Bu bölümde değişkenlerin pandas saklama tipleri ile
istatistiksel ölçüm ölçekleri karşılaştırılmaktadır.

Ölçüm ölçekleri, değişkenlerin anlamlarına ve veri
sözlüğündeki tanımlarına göre belirlenmiştir.

In [7]:
olcum_olcekleri = {
    "age": "Oran",
    "job": "Nominal",
    "marital": "Nominal",
    "education": "Ordinal",
    "default": "Nominal",
    "balance": "Nicel (aralık benzeri)",
    "housing": "Nominal",
    "loan": "Nominal",
    "contact": "Nominal",
    "day": "Takvimsel / döngüsel",
    "month": "Takvimsel / döngüsel",
    "duration": "Oran",
    "campaign": "Oran",
    "pdays": "Oran + özel kod (-1)",
    "previous": "Oran",
    "poutcome": "Nominal",
    "y": "Nominal"
}

# Her sütunun tanımlandığını kontrol et
assert set(olcum_olcekleri) == set(df.columns)

# Önceki kalite tablosuna ölçüm ölçeğini ekle
olcek_tablosu = kalite_tablosu.copy()

olcek_tablosu["Ölçüm ölçeği"] = (
    olcek_tablosu["Öznitelik"].map(olcum_olcekleri)
)

display(
    olcek_tablosu[
        ["Öznitelik", "Saklama tipi", "Ölçüm ölçeği"]
    ]
)

,Öznitelik,Saklama tipi,Ölçüm ölçeği
0,age,int64,Oran
1,job,object,Nominal
2,marital,object,Nominal
3,education,object,Ordinal
4,default,object,Nominal
5,balance,int64,Nicel (aralık benzeri)
6,housing,object,Nominal
7,loan,object,Nominal
8,contact,object,Nominal
9,day,int64,Takvimsel / döngüsel


### Yorum - Ölçüm Ölçekleri

Veri kümesindeki değişkenlerin saklama tipleri ile
istatistiksel ölçüm ölçeklerinin farklı olduğu
görülmektedir.

Örneğin education kategorik olarak saklanmasına
rağmen eğitim seviyeleri arasında sıralama
bulunduğundan ordinal olarak değerlendirilmiştir.

Month ve day değişkenleri takvimsel özellikler
taşımaktadır. Bu değişkenlere sıradan sürekli
sayısal değişkenler gibi yaklaşılması uygun
olmayabilir.

Balance değişkeni negatif ve pozitif değerler
içerebildiği için nicel, aralık benzeri olarak
tanımlanmıştır.

Pdays değişkenindeki -1 değeri önceki iletişimin
bulunmamasını göstermektedir. Dolayısıyla bu
değer, gün sayısıyla aynı ölçekte yorumlanamaz.

Duration değişkeni oran ölçeğinde olmasına rağmen
görüşme sonrasında bilindiği için tahmin modelinden
çıkarılmalıdır.

Sonuç olarak veri ön işleme yöntemleri seçilirken
yalnızca pandas veri tipleri değil, değişkenlerin
anlamları ve ölçüm ölçekleri de dikkate alınmalıdır.